# Ontario Renal Data Pipeline (All-in-One Colab Notebook)

This notebook contains the **entire data-prep pipeline in one file**.

It covers:
1. ON-Marg loading and cleaning (2016 + 2021)
2. Patient file loading and cleaning
3. Crosswalk loading (or synthetic fallback)
4. Patient-to-DAUID linkage and 2021 aggregation
5. ML base table creation
6. Deprivation-coefficient regression (true/demo labeled)
7. Synthetic 2016-2020 training generation + 2021 holdout
8. QA reports

## Important limitation (plain language)
If you do not have a real postal->DAUID crosswalk and complete 2021 ORN volume labels,
this notebook still runs but marks results as **demo-only** for validation.


## How this code is organized

- **Configuration block**: all paths, years, and file patterns.
- **Utility functions**: cleaning and file discovery.
- **Step functions**: one function per pipeline stage.
- **Orchestrator**: `run_pipeline()` calls each stage in sequence.
- **Reports**: writes schema and QA markdown files so outputs are auditable.

Each function has inline comments to explain what it does and why.


## Function Steps (Detailed)

This section lists each function and the steps it performs in plain language.

- `normalize_col_name`: Steps: 1. Lowercase and replace symbols with underscores. 2. Collapse repeated underscores.
- `normalize_dauid_value`: Steps: 1. Convert value to string. 2. Strip non-digits. 3. Return None if invalid.
- `standardize_postal_code`: Steps: 1. Uppercase and remove spaces/symbols. 2. Validate A1A1A1 format.
- `read_table`: Steps: 1. Detect file type by suffix. 2. Read CSV or Excel into pandas.
- `list_candidate_files`: Steps: 1. Scan search folders. 2. Return files matching patterns.
- `prioritize_paths`: Steps: 1. Sort candidates by location and suffix. 2. Prefer non-synthetic names.
- `discover_onmarg_files`: Steps: 1. Search raw and Downloads. 2. Pick best 2016 and 2021 file.
- `_excel_engine`: Steps: 1. Use xlrd for .xls. 2. Use default for .xlsx.
- `_select_onmarg_sheet`: Steps: 1. Try DA_YEAR sheet name. 2. Fallback to DAUID column detection.
- `load_onmarg_year`: Steps: 1. Load selected sheet. 2. Return data + metadata.
- `_pick_dauid_column`: Steps: 1. Run function-specific logic; see inline comments in code.
- `_pick_dimension_column`: Steps: 1. Score candidate columns. 2. Prefer score over quantile columns.
- `clean_onmarg`: Steps: 1. Select DAUID + 4 dimensions. 2. Normalize DAUID and numeric values. 3. Save clean CSV.
- `ensure_patient_templates`: Steps: 1. Create template CSV if missing. 2. Create tiny placeholder sample if missing.
- `_looks_like_patient_file`: Steps: 1. Read small sample. 2. Require postal + count/year signals.
- `discover_patient_file`: Steps: 1. Rank candidates. 2. Prefer real > synthetic > placeholder.
- `_pick_col`: Steps: 1. Normalize column names. 2. Match against token list.
- `load_and_clean_patients`: Steps: 1. Standardize required columns. 2. Apply defaults if missing. 3. Clean postals and numeric values.
- `_require_geopandas`: Steps: 1. Import geopandas on demand. 2. Raise clear error if missing.
- `discover_crosswalk_file`: Steps: 1. Search for crosswalk files. 2. Prefer non-synthetic.
- `discover_da_boundary_zip`: Steps: 1. Find DA boundary ZIP for year.
- `discover_points_file`: Steps: 1. Find points CSV with lat/lon/postal.
- `_pick_any_column`: Steps: 1. Run function-specific logic; see inline comments in code.
- `load_and_clean_crosswalk`: Steps: 1. Select postal + DAUID columns. 2. Normalize values. 3. Save clean CSV.
- `load_da_boundaries`: Steps: 1. Read DA polygons. 2. Filter to Ontario if possible.
- `download_geoinfo_points_file`: Steps: 1. Download public dataset if missing. 2. Filter to Ontario postal points. 3. Write to CSV.
- `build_crosswalk_from_boundaries_and_points`: Steps: 1. Create points GeoDataFrame. 2. Spatial join to DA polygons. 3. Fallback to nearest DA if needed. 4. Reduce to one DAUID per postal.
- `_hash_assign_dauid`: Steps: 1. Compute deterministic hash. 2. Assign DAUID by hash index.
- `_generate_postals`: Steps: 1. Generate synthetic postal codes.
- `generate_synthetic_crosswalk`: Steps: 1. Create demo-only postal->DAUID mapping. 2. Use observed postals if provided.
- `map_patients_to_dauid`: Steps: 1. Left join patients to crosswalk. 2. Write unmapped postal report. 3. Compute mapping rates.
- `_composite_score`: Steps: 1. Compute mean of the 4 OMI dimensions.
- `estimate_deprivation_coefficient`: Steps: 1. Regress OMI composite vs 2021 volume. 2. Compute gamma and save metadata.
- `generate_temporal_training_data`: Steps: 1. Interpolate 2016–2021 OMI. 2. Backcast 2016–2020 volumes. 3. Save train/validation files.
- `_fmt_columns`: Steps: 1. Render column list for markdown.
- `write_schema_summary`: Steps: 1. Write input schema report.
- `write_qa_report`: Steps: 1. Write mapping and output summary.
- `write_deprivation_coefficient_report`: Steps: 1. Write coefficient quality report.
- `run_pipeline`: Steps: 1. Run all steps in order. 2. Record warnings/blockers. 3. Write QA reports.


## Function-by-function guide (plain English)

- `normalize_col_name`: cleans messy column names so matching works even if names differ a bit.
- `standardize_postal_code`: converts postal values to standard format (like `A1A1A1`) and drops invalid values.
- `discover_onmarg_files`: finds 2016 and 2021 ON-Marg files automatically.
- `clean_onmarg`: picks DAUID + 4 deprivation dimensions and renames them to canonical names.
- `discover_patient_file`: selects the best available patient file (real first, then synthetic, then placeholder).
- `load_and_clean_patients`: standardizes patient columns (`postal_code`, `patient_count`, `year`, `source_flag`).
- `discover_crosswalk_file` / `load_and_clean_crosswalk`: loads postal-code to DAUID mapping if available.
- `generate_synthetic_crosswalk`: creates a demo-only fallback mapping when real crosswalk is missing.
- `map_patients_to_dauid`: links patient rows to DAUID and writes unmapped postal QA file.
- `aggregate_patient_volume`: sums patient counts to `patient_volume_2021` per DAUID.
- `build_ml_base_2021`: merges ON-Marg features with the DAUID-level target.
- `estimate_deprivation_coefficient`: computes the coefficient used by synthetic backcasting logic.
- `generate_temporal_training_data`: creates synthetic 2016-2020 training rows and keeps 2021 as holdout.
- `write_schema_summary` / `write_qa_report`: writes human-readable QA documentation.
- `run_pipeline`: runs all steps in order and records blockers/warnings.


In [ ]:
# Optional in Colab if dependencies are missing:
# !pip -q install pandas numpy openpyxl xlrd


In [ ]:
from __future__ import annotations

import fnmatch
import hashlib
import re
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

# ============================================================
# 1) Configuration
# ============================================================
# For Colab + Drive, point this to your folder.
# Example: Path('/content/drive/MyDrive/group11_courseproj')
PROJECT_ROOT = Path('/content/drive/MyDrive/group11_courseproj')

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

RAW_ONMARG_DIR = RAW_DIR / 'onmarg'
RAW_CROSSWALK_DIR = RAW_DIR / 'crosswalk'
RAW_PATIENTS_DIR = RAW_DIR / 'patients'

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
QA_DIR = OUTPUTS_DIR / 'qa'

# Fallback search location if files are not inside project folders.
DOWNLOADS_DIR = Path.home() / 'Downloads'

TARGET_YEAR = 2021
SYNTHETIC_START_YEAR = 2016
SYNTHETIC_END_YEAR = 2021
SYNTHETIC_RANDOM_SEED = 5130

# If True, notebook can create synthetic fallback assets so demo can run end-to-end.
ALLOW_SYNTHETIC_WORKAROUNDS = True

# Ontario province code used in StatsCan boundary files.
ONTARIO_PRUID = '35'

# Optional public dataset used to build approximate postal->DAUID mapping.
# Source: https://github.com/djbelieny/geoinfo-dataset (MIT license)
GEOINFO_DATASET_URL = 'https://github.com/djbelieny/geoinfo-dataset/raw/master/full_dataset_csv.zip'
GEOINFO_ZIP_NAME = 'geoinfo_full_dataset_csv.zip'
GEOINFO_POINTS_FILE = 'geoinfo_points_canada_on.csv'

CANONICAL_ONMARG_COLUMNS = [
    'DAUID',
    'material_deprivation',
    'residential_instability',
    'dependency',
    'ethnic_concentration',
]

ONMARG_DIMENSION_PATTERNS = {
    'material_deprivation': ['deprivation', 'material_resources', 'materialresource'],
    'residential_instability': ['instability', 'households_dwellings', 'householddwellings'],
    'dependency': ['dependency', 'age_labourforce', 'age_laborforce', 'agelabourforce'],
    'ethnic_concentration': ['ethniccon', 'ethnic_concentration', 'racialized_nc_pop', 'racializedncpop'],
}

ONMARG_FILE_PATTERNS = {
    2016: ['*2016*.xlsx', '*2016*.xls', '*index-on-marg*2016*', '*on-marg*2016*'],
    2021: ['*2021*.xlsx', '*2021*.xls', '*index-on-marg*2021*', '*on-marg*2021*', '*index-on-marg 1.xlsx'],
}

# DA boundary shapefile ZIPs (StatsCan LDA files).
DA_BOUNDARY_FILE_PATTERNS = {
    2016: ['*lda_000a16a*.zip', '*lda_000a*16*.zip'],
    2021: ['*lda_000a21a*.zip', '*lda_000a*21*.zip'],
}

# Crosswalk and address point file patterns.
CROSSWALK_FILE_PATTERNS = [
    '*crosswalk*.csv', '*crosswalk*.xlsx', '*crosswalk*.xls',
    '*postal*da*.*', '*dauid*postal*.*', '*pccf*.*'
]

POINTS_FILE_PATTERNS = [
    '*openaddresses*canada*.csv',
    '*open-addresses*canada*.csv',
    '*openaddresses*.csv',
    '*geoinfo*points*canada*on*.csv',
    '*postal*points*.csv',
]

# Column-name token lists.
CROSSWALK_POSTAL_PATTERNS = ['postal', 'postcode', 'pc6', 'fsa_ldu']
CROSSWALK_DAUID_PATTERNS = ['dauid', 'da_uid']

PATIENT_POSTAL_PATTERNS = ['postal', 'postcode', 'pc6']
PATIENT_COUNT_PATTERNS = ['patient_count', 'count', 'volume', 'patients', 'n_patients', 'num_patients']
PATIENT_YEAR_PATTERNS = ['year', 'fiscal_year', 'calendar_year']

# Point dataset patterns (for spatial join if PCCF not available).
POINT_LAT_PATTERNS = ['lat', 'latitude', 'y']
POINT_LON_PATTERNS = ['lon', 'longitude', 'lng', 'x']
POINT_POSTAL_PATTERNS = ['postal', 'postcode', 'pc6']

TEMPLATE_FILE = RAW_PATIENTS_DIR / 'patient_2021_template.csv'
PLACEHOLDER_FILE = RAW_PATIENTS_DIR / 'patient_2021_placeholder.csv'
UNMAPPED_FILE = QA_DIR / 'unmapped_postal_codes.csv'

for folder in [RAW_ONMARG_DIR, RAW_CROSSWALK_DIR, RAW_PATIENTS_DIR, INTERIM_DIR, PROCESSED_DIR, QA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# =========================
# 2) Utility functions
# =========================


def ensure_dir(path: Path) -> None:
    # Create a directory if it does not already exist.
    path.mkdir(parents=True, exist_ok=True)


def normalize_col_name(name: str) -> str:
    value = re.sub(r'[^a-zA-Z0-9]+', '_', str(name).strip().lower())
    return re.sub(r'_+', '_', value).strip('_')


def normalize_dauid_value(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    if text.endswith('.0'):
        text = text[:-2]
    digits = re.sub(r'\D', '', text)
    return digits if digits else None


def standardize_postal_code(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = re.sub(r'[^A-Za-z0-9]', '', str(value).upper())
    if len(text) != 6:
        return None
    if not re.fullmatch(r'[A-Z]\d[A-Z]\d[A-Z]\d', text):
        return None
    return text


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file format: {path}')


def list_candidate_files(search_roots: Iterable[Path], patterns: Iterable[str]) -> list[Path]:
    normalized_patterns = [p.lower() for p in patterns]
    seen: set[Path] = set()
    out: list[Path] = []
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in root.iterdir():
            if not candidate.is_file():
                continue
            lower_name = candidate.name.lower()
            if any(fnmatch.fnmatch(lower_name, pat) for pat in normalized_patterns):
                if candidate not in seen:
                    out.append(candidate)
                    seen.add(candidate)
    return out


def prioritize_paths(candidates: list[Path], preferred_suffixes: tuple[str, ...] = ()) -> list[Path]:
    suffix_rank = {s.lower(): i for i, s in enumerate(preferred_suffixes)}
    def score(path: Path) -> tuple[int, int, int, str]:
        in_repo = 0 if str(path).startswith(str(PROJECT_ROOT)) else 1
        suffix_score = suffix_rank.get(path.suffix.lower(), len(suffix_rank))
        return (in_repo, suffix_score, int('synthetic' in path.name.lower()), path.name.lower())
    return sorted(candidates, key=score)


# Patient template files are always created so pipeline stays usable.
def ensure_patient_templates() -> dict[str, object]:
    # Create template CSV and tiny placeholder sample if missing.
    ensure_dir(RAW_PATIENTS_DIR)
    template_created = False
    placeholder_created = False

    if not TEMPLATE_FILE.exists():
        pd.DataFrame(columns=['postal_code', 'patient_count', 'year', 'source_flag']).to_csv(TEMPLATE_FILE, index=False)
        template_created = True

    if not PLACEHOLDER_FILE.exists():
        pd.DataFrame([
            {'postal_code': 'M5V3L9', 'patient_count': 3, 'year': 2021, 'source_flag': 'placeholder'},
            {'postal_code': 'K1A0B1', 'patient_count': 2, 'year': 2021, 'source_flag': 'placeholder'},
            {'postal_code': 'L6H1M2', 'patient_count': 1, 'year': 2021, 'source_flag': 'placeholder'},
        ]).to_csv(PLACEHOLDER_FILE, index=False)
        placeholder_created = True

    return {
        'template_path': str(TEMPLATE_FILE),
        'placeholder_path': str(PLACEHOLDER_FILE),
        'template_created': template_created,
        'placeholder_created': placeholder_created,
    }


In [ ]:
# =========================
# 3) Data loading + cleaning helpers
# =========================

def discover_onmarg_files() -> dict[int, Path | None]:
    roots = [RAW_ONMARG_DIR, DOWNLOADS_DIR]
    found = {2016: None, 2021: None}
    for year in [2016, 2021]:
        c = list_candidate_files(roots, ONMARG_FILE_PATTERNS[year])
        r = prioritize_paths(c, preferred_suffixes=('.xlsx', '.xls'))
        found[year] = r[0] if r else None
    return found


def _excel_engine(path: Path) -> str | None:
    return 'xlrd' if path.suffix.lower() == '.xls' else None


def _select_onmarg_sheet(path: Path, year: int) -> str:
    xls = pd.ExcelFile(path, engine=_excel_engine(path))
    for sn in xls.sheet_names:
        if sn.lower() == f'da_{year}'.lower():
            return sn
    for sn in xls.sheet_names:
        n = normalize_col_name(sn)
        if 'da' in n and str(year) in n:
            return sn
    for sn in xls.sheet_names:
        probe = pd.read_excel(path, sheet_name=sn, nrows=0, engine=_excel_engine(path))
        if any('dauid' in normalize_col_name(c) for c in probe.columns):
            return sn
    raise RuntimeError(f'No DA-level sheet found in {path}')


def load_onmarg_year(path: Path, year: int) -> tuple[pd.DataFrame, dict[str, Any]]:
    sheet = _select_onmarg_sheet(path, year)
    df = pd.read_excel(path, sheet_name=sheet, engine=_excel_engine(path))
    return df, {
        'year': year,
        'source_path': str(path),
        'sheet_name': sheet,
        'detected_columns': [str(c) for c in df.columns],
        'row_count': int(len(df)),
    }


def _pick_dauid_column(columns: list[str]) -> str | None:
    for col in columns:
        n = normalize_col_name(col)
        if n == 'dauid' or 'dauid' in n:
            return col
    return None


def _pick_dimension_column(columns: list[str], tokens: list[str]) -> str | None:
    best_col = None
    best_score = (-1, -1)
    for col in columns:
        n = normalize_col_name(col)
        hits = sum(1 for t in tokens if t in n)
        if hits == 0:
            continue
        is_q = 1 if n.endswith('_q') or '_q_' in n else 0
        score = (hits, -is_q)
        if score > best_score:
            best_score = score
            best_col = col
    return best_col


def clean_onmarg(df: pd.DataFrame, year: int) -> tuple[pd.DataFrame, dict[str, str]]:
    cols = [str(c) for c in df.columns]
    dauid_col = _pick_dauid_column(cols)
    if not dauid_col:
        raise RuntimeError(f'Could not find DAUID column for ON-Marg {year}')

    selected = {'DAUID': dauid_col}
    for cname, pats in ONMARG_DIMENSION_PATTERNS.items():
        c = _pick_dimension_column(cols, pats)
        if not c:
            raise RuntimeError(f'Missing ON-Marg dimension {cname} for {year}')
        selected[cname] = c

    out = df[[
        selected['DAUID'],
        selected['material_deprivation'],
        selected['residential_instability'],
        selected['dependency'],
        selected['ethnic_concentration'],
    ]].copy()
    out.columns = CANONICAL_ONMARG_COLUMNS
    out['DAUID'] = out['DAUID'].apply(normalize_dauid_value)
    out = out.dropna(subset=['DAUID']).drop_duplicates(subset=['DAUID']).reset_index(drop=True)

    for c in CANONICAL_ONMARG_COLUMNS[1:]:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    out_path = INTERIM_DIR / f'onmarg_{year}_clean.csv'
    out.to_csv(out_path, index=False)
    return out, {
        'clean_output_path': str(out_path),
        'selected_to_canonical_mapping': {v: k for k, v in selected.items()}
    }


def ensure_patient_templates() -> dict[str, Any]:
    created_template = False
    created_placeholder = False

    if not TEMPLATE_FILE.exists():
        pd.DataFrame(columns=['postal_code', 'patient_count', 'year', 'source_flag']).to_csv(TEMPLATE_FILE, index=False)
        created_template = True

    if not PLACEHOLDER_FILE.exists():
        pd.DataFrame([
            {'postal_code': 'M5V3L9', 'patient_count': 3, 'year': 2021, 'source_flag': 'placeholder'},
            {'postal_code': 'K1A0B1', 'patient_count': 2, 'year': 2021, 'source_flag': 'placeholder'},
        ]).to_csv(PLACEHOLDER_FILE, index=False)
        created_placeholder = True

    return {
        'template_path': str(TEMPLATE_FILE),
        'placeholder_path': str(PLACEHOLDER_FILE),
        'template_created': created_template,
        'placeholder_created': created_placeholder,
    }


def _looks_like_patient_file(path: Path) -> bool:
    try:
        probe = read_table(path).head(5)
    except Exception:
        return False
    ncols = [normalize_col_name(c) for c in probe.columns]
    has_postal = any(any(tok in c for tok in PATIENT_POSTAL_PATTERNS) for c in ncols)
    has_signal = any(any(tok in c for tok in (PATIENT_COUNT_PATTERNS + PATIENT_YEAR_PATTERNS + ['visit', 'subject', 'patient'])) for c in ncols)
    return has_postal and has_signal


def discover_patient_file() -> tuple[Path, str]:
    candidates = list_candidate_files([RAW_PATIENTS_DIR, DOWNLOADS_DIR], PATIENT_FILE_PATTERNS)
    ranked = prioritize_paths(candidates, preferred_suffixes=('.csv', '.xlsx', '.xls'))

    real, synthetic, placeholder = [], [], []
    for p in ranked:
        if not _looks_like_patient_file(p):
            continue
        name = p.name.lower()
        if 'template' in name:
            continue
        if 'placeholder' in name:
            placeholder.append(p)
        elif 'synthetic' in name:
            synthetic.append(p)
        else:
            real.append(p)

    if real:
        return real[0], 'real'
    if synthetic:
        return synthetic[0], 'synthetic'
    if placeholder:
        return placeholder[0], 'placeholder'
    return PLACEHOLDER_FILE, 'placeholder'


def _pick_col(columns: list[str], patterns: list[str]) -> str | None:
    for col in columns:
        n = normalize_col_name(col)
        if any(p in n for p in patterns):
            return col
    return None


def load_and_clean_patients(path: Path, source_type: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    raw = read_table(path)
    cols = [str(c) for c in raw.columns]

    postal_col = _pick_col(cols, PATIENT_POSTAL_PATTERNS)
    if not postal_col:
        raise RuntimeError('Could not detect postal column in patient data')

    count_col = _pick_col(cols, PATIENT_COUNT_PATTERNS)
    year_col = _pick_col(cols, PATIENT_YEAR_PATTERNS)
    flag_col = _pick_col(cols, ['source_flag'])

    out = raw.rename(columns={postal_col: 'postal_code'}).copy()
    assumptions = []

    if count_col:
        out = out.rename(columns={count_col: 'patient_count'})
    else:
        out['patient_count'] = 1
        assumptions.append('patient_count column missing; defaulted each row to 1')

    if year_col:
        out = out.rename(columns={year_col: 'year'})
    else:
        out['year'] = TARGET_YEAR
        assumptions.append(f'year column missing; defaulted all rows to {TARGET_YEAR}')

    if flag_col:
        out = out.rename(columns={flag_col: 'source_flag'})
    else:
        out['source_flag'] = source_type

    out['postal_code'] = out['postal_code'].apply(standardize_postal_code)
    out['patient_count'] = pd.to_numeric(out['patient_count'], errors='coerce')
    out['year'] = pd.to_numeric(out['year'], errors='coerce')

    out = out[['postal_code', 'patient_count', 'year', 'source_flag']].copy()
    out = out.dropna(subset=['postal_code', 'patient_count', 'year']).copy()
    out = out[out['patient_count'] >= 0].copy()
    out['year'] = out['year'].astype(int)

    out_path = INTERIM_DIR / 'patients_clean.csv'
    out.to_csv(out_path, index=False)

    return out, {
        'source_path': str(path),
        'source_type': source_type,
        'used_real_patient_file': source_type == 'real',
        'used_synthetic_patient_file': source_type == 'synthetic',
        'detected_columns': cols,
        'row_count': int(len(out)),
        'assumptions': assumptions,
        'output_path': str(out_path),
    }


In [ ]:
# =========================
# 4) Crosswalk + mapping + outputs
# =========================

# Note: The boundary-based crosswalk requires geopandas.
# If you need it in Colab, install via:
# !pip -q install geopandas shapely pyogrio


def _require_geopandas():
    # Import geopandas only when needed to keep dependencies optional.
    try:
        import geopandas as gpd  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            'geopandas is required for boundary-based crosswalk. '
            'Install with: pip install geopandas shapely pyogrio'
        ) from exc
    return gpd


def discover_crosswalk_file() -> Path | None:
    # Find real crosswalk file if it exists (CSV/XLSX preferred).
    c = list_candidate_files([RAW_CROSSWALK_DIR, DOWNLOADS_DIR], CROSSWALK_FILE_PATTERNS)
    r = prioritize_paths(c, preferred_suffixes=('.csv', '.xlsx', '.xls'))
    # Prefer non-synthetic if both exist.
    r = sorted(r, key=lambda p: ('synthetic' in p.name.lower(), p.name.lower()))
    return r[0] if r else None


def discover_da_boundary_zip(year: int) -> Path | None:
    # Find StatsCan DA boundary ZIP (LDA) for a given year.
    c = list_candidate_files([RAW_CROSSWALK_DIR, DOWNLOADS_DIR], DA_BOUNDARY_FILE_PATTERNS[year])
    r = prioritize_paths(c, preferred_suffixes=('.zip',))
    return r[0] if r else None


def discover_points_file() -> Path | None:
    # Find points CSV with lat/lon/postal columns.
    roots = [RAW_CROSSWALK_DIR, RAW_PATIENTS_DIR, DOWNLOADS_DIR]
    c = list_candidate_files(roots, POINTS_FILE_PATTERNS)
    r = prioritize_paths(c, preferred_suffixes=('.csv',))
    return r[0] if r else None


def _pick_any_column(columns: list[str], patterns: list[str]) -> str | None:
    # Pick the first column that matches any token in patterns.
    for col in columns:
        n = normalize_col_name(col)
        if any(tok in n for tok in patterns):
            return col
    return None


def load_and_clean_crosswalk(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Load crosswalk and standardize to postal_code + DAUID.
    raw = read_table(path)
    cols = [str(c) for c in raw.columns]

    postal_col = _pick_any_column(cols, CROSSWALK_POSTAL_PATTERNS)
    dauid_col = _pick_any_column(cols, CROSSWALK_DAUID_PATTERNS)
    if not postal_col or not dauid_col:
        raise RuntimeError('Crosswalk found but postal/DAUID columns not detected')

    out = raw[[postal_col, dauid_col]].copy()
    out.columns = ['postal_code', 'DAUID']
    out['postal_code'] = out['postal_code'].apply(standardize_postal_code)
    out['DAUID'] = out['DAUID'].apply(normalize_dauid_value)
    out = out.dropna(subset=['postal_code', 'DAUID']).drop_duplicates().reset_index(drop=True)

    out_path = INTERIM_DIR / 'crosswalk_clean.csv'
    out.to_csv(out_path, index=False)

    return out, {
        'source_path': str(path),
        'detected_columns': cols,
        'row_count': int(len(out)),
        'output_path': str(out_path),
        'is_synthetic': 'synthetic' in path.name.lower(),
        'is_approximate': False,
    }


def load_da_boundaries(path: Path) -> tuple[Any, dict[str, Any]]:
    # Load DA boundary polygons and try to filter to Ontario.
    gpd = _require_geopandas()
    gdf = gpd.read_file(f'zip://{path}')

    cols = [str(c) for c in gdf.columns]
    dauid_col = _pick_any_column(cols, ['dauid', 'da_uid'])
    if not dauid_col:
        raise RuntimeError('DA boundary file does not include a DAUID-like column')

    # Try to filter to Ontario if a province column is present.
    pruid_col = _pick_any_column(cols, ['pruid'])
    prname_col = _pick_any_column(cols, ['prname', 'province', 'prov'])

    filtered = gdf
    if pruid_col:
        filtered = gdf[gdf[pruid_col].astype(str) == ONTARIO_PRUID].copy()
    elif prname_col:
        filtered = gdf[gdf[prname_col].astype(str).str.lower().str.contains('ontario')].copy()

    if filtered.empty:
        # Fallback: if filtering removes everything, keep all rows.
        filtered = gdf.copy()

    meta = {
        'source_path': str(path),
        'row_count': int(len(filtered)),
        'dauid_column': dauid_col,
        'filtered_to_ontario': not filtered.equals(gdf),
    }
    return filtered, meta


def download_geoinfo_points_file() -> Path | None:
    # Download and filter geoinfo dataset to Ontario postal points (if missing).
    ensure_dir(RAW_CROSSWALK_DIR)

    zip_path = RAW_CROSSWALK_DIR / GEOINFO_ZIP_NAME
    out_path = RAW_CROSSWALK_DIR / GEOINFO_POINTS_FILE

    if out_path.exists():
        return out_path

    if not zip_path.exists():
        try:
            urllib.request.urlretrieve(GEOINFO_DATASET_URL, zip_path)
        except Exception:
            return None

    # Find the CSV inside the ZIP (skip __MACOSX).
    with zipfile.ZipFile(zip_path) as zf:
        csv_candidates = [n for n in zf.namelist() if n.lower().endswith('.csv') and '__macosx' not in n.lower()]
        if not csv_candidates:
            raise RuntimeError('No CSV found in geoinfo ZIP archive')
        csv_name = csv_candidates[0]

    wrote_any = False

    # Read the zipped CSV in chunks and filter to Canada + Ontario.
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(csv_name) as fh:
            for chunk in pd.read_csv(fh, chunksize=200000):
                # Normalize column names for robust matching.
                cols = {normalize_col_name(c): c for c in chunk.columns}
                required = ['countryiso', 'stateiso', 'zipcode', 'latitude', 'longitude']
                if not all(r in cols for r in required):
                    raise RuntimeError('Geoinfo dataset schema not recognized for postal points')

                work = chunk[[cols['countryiso'], cols['stateiso'], cols['zipcode'], cols['latitude'], cols['longitude']]].copy()
                work.columns = ['countryiso', 'stateiso', 'postal_code', 'lat', 'lon']

                work['countryiso'] = work['countryiso'].astype(str).str.upper()
                work = work[work['countryiso'] == 'CA']

                if work.empty:
                    continue

                work['stateiso'] = work['stateiso'].astype(str).str.upper()
                work = work[work['stateiso'] == 'ON']
                if work.empty:
                    continue

                work['postal_code'] = work['postal_code'].apply(standardize_postal_code)
                work = work.dropna(subset=['postal_code', 'lat', 'lon'])

                if work.empty:
                    continue

                work[['postal_code', 'lat', 'lon']].to_csv(out_path, mode='a', index=False, header=not wrote_any)
                wrote_any = True

    return out_path if wrote_any else None


def build_crosswalk_from_boundaries_and_points(boundary_zip: Path, points_file: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Build an approximate postal_code -> DAUID crosswalk via spatial join.
    gpd = _require_geopandas()

    # 1) Load DA polygons.
    da_gdf, da_meta = load_da_boundaries(boundary_zip)
    dauid_col = da_meta['dauid_column']

    if da_gdf.crs is None:
        raise RuntimeError('DA boundary file has no CRS; cannot do spatial join safely.')

    # 2) Load address points (must include lat, lon, postal).
    raw = pd.read_csv(points_file)
    cols = [str(c) for c in raw.columns]

    lat_col = _pick_any_column(cols, POINT_LAT_PATTERNS)
    lon_col = _pick_any_column(cols, POINT_LON_PATTERNS)
    postal_col = _pick_any_column(cols, POINT_POSTAL_PATTERNS)

    if not lat_col or not lon_col or not postal_col:
        raise RuntimeError('Point file missing lat/lon/postal columns required for spatial join')

    work = raw[[lat_col, lon_col, postal_col]].copy()
    work.columns = ['lat', 'lon', 'postal_code']

    work['postal_code'] = work['postal_code'].apply(standardize_postal_code)
    work = work.dropna(subset=['postal_code', 'lat', 'lon'])

    # 3) Convert to GeoDataFrame (WGS84), then project to DA CRS.
    points = gpd.GeoDataFrame(
        work,
        geometry=gpd.points_from_xy(work['lon'], work['lat']),
        crs='EPSG:4326',
    )
    points = points.to_crs(da_gdf.crs)

    # 4) Spatial join: which DA each point falls inside (exact match).
    joined = gpd.sjoin(points, da_gdf[[dauid_col, 'geometry']], how='left', predicate='within')

    # If DAUID column name changed during join, recover it for the joined frame only.
    joined_dauid_col = dauid_col
    if joined_dauid_col not in joined.columns:
        candidates = [c for c in joined.columns if 'dauid' in normalize_col_name(c)]
        if candidates:
            joined_dauid_col = candidates[0]
        else:
            raise RuntimeError('DAUID column missing after spatial join')

    # 5) If any points are still unmatched, use nearest DA centroid as a fallback.
    unmatched = joined[joined[joined_dauid_col].isna()].copy()
    nearest_used = 0
    if not unmatched.empty:
        # For nearest, use a projected CRS if we are in geographic degrees.
        da_for_nearest = da_gdf
        pts_for_nearest = unmatched
        if da_gdf.crs and da_gdf.crs.is_geographic:
            da_for_nearest = da_gdf.to_crs('EPSG:3347')
            pts_for_nearest = unmatched.to_crs('EPSG:3347')

        # Drop index_right if it exists from the previous spatial join.
        if 'index_right' in pts_for_nearest.columns:
            pts_for_nearest = pts_for_nearest.drop(columns=['index_right'])

        da_centroids = da_for_nearest.copy()
        da_centroids['geometry'] = da_centroids.geometry.centroid

        nearest = gpd.sjoin_nearest(
            pts_for_nearest,
            da_centroids[[dauid_col, 'geometry']],
            how='left',
            distance_col='nearest_dist',
        )

        # Recover DAUID column name if it changed during nearest join.
        nearest_dauid_col = dauid_col
        if nearest_dauid_col not in nearest.columns:
            candidates = [c for c in nearest.columns if 'dauid' in normalize_col_name(c)]
            if candidates:
                nearest_dauid_col = candidates[0]
            else:
                raise RuntimeError('DAUID column missing after nearest join')

        # Map nearest DAUIDs back onto the original joined frame.
        joined.loc[unmatched.index, joined_dauid_col] = nearest[nearest_dauid_col].values
        nearest_used = int(len(unmatched))

    # 6) Reduce to one DAUID per postal code (most common DAUID).
    joined = joined[['postal_code', joined_dauid_col]].rename(columns={joined_dauid_col: 'DAUID'})
    counts = joined.groupby(['postal_code', 'DAUID']).size().reset_index(name='n')

    # Choose the DAUID with the most points for each postal code.
    idx = counts.sort_values(['postal_code', 'n'], ascending=[True, False]).groupby('postal_code').head(1)
    crosswalk = idx[['postal_code', 'DAUID']].copy().reset_index(drop=True)

    # Count ambiguous postal codes (multiple DAUIDs seen).
    ambiguous = counts.groupby('postal_code').size()
    ambiguous_count = int((ambiguous > 1).sum())

    out_path = RAW_CROSSWALK_DIR / 'crosswalk_approx_from_boundary_points.csv'
    crosswalk.to_csv(out_path, index=False)

    meta = {
        'source_path': str(out_path),
        'row_count': int(len(crosswalk)),
        'output_path': str(out_path),
        'is_synthetic': False,
        'is_approximate': True,
        'approx_method': 'boundary_point_join_with_nearest_fallback',
        'boundary_source': str(boundary_zip),
        'points_source': str(points_file),
        'ambiguous_postal_codes': ambiguous_count,
        'nearest_fallback_rows': nearest_used,
    }
    return crosswalk, meta


def _hash_assign_dauid(postal_code: str, dauids: list[str]) -> str:
    # Deterministic hash assignment used for synthetic fallback.
    digest = hashlib.sha256(f"{postal_code}|{SYNTHETIC_RANDOM_SEED}".encode('utf-8')).hexdigest()
    idx = int(digest[:12], 16) % len(dauids)
    return dauids[idx]


def _generate_postals(n: int) -> list[str]:
    # Generate fake postal codes so demo pipeline can run without real crosswalk.
    rng = np.random.default_rng(SYNTHETIC_RANDOM_SEED)
    letters = np.array(list('ABCEGHJKLMNPRSTVXY'))
    codes = set()
    while len(codes) < n:
        chunk = max(1000, n - len(codes))
        a = rng.choice(letters, size=chunk)
        b = rng.integers(0, 10, size=chunk).astype(str)
        c = rng.choice(letters, size=chunk)
        d = rng.integers(0, 10, size=chunk).astype(str)
        e = rng.choice(letters, size=chunk)
        f = rng.integers(0, 10, size=chunk).astype(str)
        for i in range(chunk):
            codes.add(f"{a[i]}{b[i]}{c[i]}{d[i]}{e[i]}{f[i]}")
            if len(codes) == n:
                break
    return sorted(codes)


def generate_synthetic_crosswalk(onmarg_2021: pd.DataFrame, observed_postals: pd.Series | None = None):
    # Synthetic fallback when no real or approximate crosswalk exists.
    dauids = sorted(onmarg_2021['DAUID'].dropna().astype(str).unique().tolist())
    strategy = 'generated_postals_one_to_one'
    observed_count = 0

    if observed_postals is not None:
        clean = [standardize_postal_code(v) for v in observed_postals.tolist()]
        unique_postals = sorted({v for v in clean if v})
        if unique_postals:
            strategy = 'patient_postal_hash_assignment'
            observed_count = len(unique_postals)
            out = pd.DataFrame({
                'postal_code': unique_postals,
                'DAUID': [_hash_assign_dauid(pc, dauids) for pc in unique_postals],
            })
        else:
            out = pd.DataFrame({'postal_code': _generate_postals(len(dauids)), 'DAUID': dauids})
    else:
        out = pd.DataFrame({'postal_code': _generate_postals(len(dauids)), 'DAUID': dauids})

    out_path = RAW_CROSSWALK_DIR / 'crosswalk_synthetic_pc_to_dauid.csv'
    out.to_csv(out_path, index=False)

    return out, {
        'source_path': str(out_path),
        'row_count': int(len(out)),
        'output_path': str(out_path),
        'is_synthetic': True,
        'is_approximate': False,
        'synthetic_strategy': strategy,
        'observed_postal_count': observed_count,
    }


def map_patients_to_dauid(patients_df: pd.DataFrame, crosswalk_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, object]]:
    # Join patient rows to DAUID and write unmapped postal QA file.
    ensure_dir(QA_DIR)
    mapped = patients_df.merge(crosswalk_df, on='postal_code', how='left')

    unmapped = (
        mapped[mapped['DAUID'].isna()]
        .groupby('postal_code', as_index=False)
        .agg(unmapped_rows=('postal_code', 'size'), unmapped_patient_count=('patient_count', 'sum'))
        .sort_values('unmapped_patient_count', ascending=False)
    )
    unmapped.to_csv(UNMAPPED_FILE, index=False)

    total_rows = int(len(mapped))
    mapped_rows = int(mapped['DAUID'].notna().sum())
    total_patient_count = float(mapped['patient_count'].sum()) if total_rows else 0.0
    mapped_patient_count = float(mapped.loc[mapped['DAUID'].notna(), 'patient_count'].sum()) if total_rows else 0.0

    return mapped, {
        'total_rows': total_rows,
        'mapped_rows': mapped_rows,
        'row_mapping_rate': (mapped_rows / total_rows) if total_rows else 0.0,
        'total_patient_count': total_patient_count,
        'mapped_patient_count': mapped_patient_count,
        'patient_count_mapping_rate': (mapped_patient_count / total_patient_count) if total_rows else 0.0,
        'unmapped_postal_codes': int(len(unmapped)),
        'unmapped_output_path': str(UNMAPPED_FILE),
    }


In [ ]:
# =========================
# 5) Coefficient + synthetic history
# =========================

def _composite_score(df: pd.DataFrame) -> pd.Series:
    return df[['material_deprivation', 'residential_instability', 'dependency', 'ethnic_concentration']].mean(axis=1)


def estimate_deprivation_coefficient(onmarg_2021: pd.DataFrame, patient_volume_df: pd.DataFrame, *, is_true: bool, reason_not_true: str):
    merged = onmarg_2021.merge(patient_volume_df, on='DAUID', how='left')
    merged['patient_volume_2021'] = merged['patient_volume_2021'].fillna(0)
    merged['omi_composite_2021'] = _composite_score(merged)

    work = merged.dropna(subset=['omi_composite_2021', 'patient_volume_2021']).copy()
    x = work['omi_composite_2021'].astype(float).to_numpy()
    y = work['patient_volume_2021'].astype(float).to_numpy()

    if len(x) < 2 or float(np.var(x)) == 0.0:
        slope = 0.0
        intercept = float(np.mean(y)) if len(y) else 0.0
    else:
        x_mean = float(np.mean(x))
        y_mean = float(np.mean(y))
        slope = float(np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2))
        intercept = y_mean - slope * x_mean

    mean_volume = float(np.mean(y)) if len(y) else 0.0
    gamma = abs(slope) / max(mean_volume, 1.0)
    gamma = max(0.0, min(gamma, 1.0))

    out_path = INTERIM_DIR / 'deprivation_coefficient_estimate.csv'
    pd.DataFrame([{
        'coefficient_type': 'true' if is_true else 'demo',
        'gamma_estimate': gamma,
        'slope': slope,
        'intercept': intercept,
        'rows_used': int(len(work)),
        'mean_patient_volume_2021': mean_volume,
        'reason_not_true': reason_not_true,
    }]).to_csv(out_path, index=False)

    return gamma, {
        'coefficient_type': 'true' if is_true else 'demo',
        'gamma_estimate': gamma,
        'slope': slope,
        'intercept': intercept,
        'rows_used': int(len(work)),
        'reason_not_true': reason_not_true,
        'output_path': str(out_path),
    }


def generate_temporal_training_data(onmarg_2016: pd.DataFrame, onmarg_2021: pd.DataFrame, patient_volume_df: pd.DataFrame, gamma: float):
    features = ['material_deprivation', 'residential_instability', 'dependency', 'ethnic_concentration']

    b16 = onmarg_2016[['DAUID'] + features].rename(columns={c: f'{c}_2016' for c in features})
    b21 = onmarg_2021[['DAUID'] + features].rename(columns={c: f'{c}_2021' for c in features})

    merged = b21.merge(b16, on='DAUID', how='inner').merge(patient_volume_df, on='DAUID', how='left')
    merged['patient_volume_2021'] = merged['patient_volume_2021'].fillna(0).astype(float)

    rng = np.random.default_rng(SYNTHETIC_RANDOM_SEED)
    rows = []
    year_span = max(SYNTHETIC_END_YEAR - SYNTHETIC_START_YEAR, 1)

    for _, row in merged.iterrows():
        dauid = str(row['DAUID'])
        p2021 = float(row['patient_volume_2021'])
        f16 = {c: float(row[f'{c}_2016']) for c in features}
        f21 = {c: float(row[f'{c}_2021']) for c in features}
        omi16 = float(np.mean(list(f16.values())))
        omi21 = float(np.mean(list(f21.values())))

        for year in range(SYNTHETIC_START_YEAR, SYNTHETIC_END_YEAR + 1):
            frac = (year - SYNTHETIC_START_YEAR) / year_span
            ft = {c: f16[c] + frac * (f21[c] - f16[c]) for c in features}
            omi_t = float(np.mean(list(ft.values())))
            expected = max(0.05, float(p2021 * (1 - gamma * (omi21 - omi_t))))

            if year < TARGET_YEAR:
                volume = int(rng.poisson(expected))
                source = 'synthetic_backcast'
            else:
                volume = int(round(p2021))
                source = 'observed_current_2021_source'

            rows.append({
                'DAUID': dauid,
                'year': year,
                'material_deprivation': ft['material_deprivation'],
                'residential_instability': ft['residential_instability'],
                'dependency': ft['dependency'],
                'ethnic_concentration': ft['ethnic_concentration'],
                'omi_composite': omi_t,
                'expected_count': expected,
                'patient_volume': volume,
                'volume_source': source,
            })

    history = pd.DataFrame(rows)

    history_path = INTERIM_DIR / 'synthetic_patient_volume_by_dauid_2016_2021.csv'
    train_path = PROCESSED_DIR / 'ml_train_synthetic_2016_2020.csv'
    val_path = PROCESSED_DIR / 'ml_validation_2021_observed.csv'

    history.to_csv(history_path, index=False)
    history[history['year'] < TARGET_YEAR].to_csv(train_path, index=False)
    val = history[history['year'] == TARGET_YEAR].rename(columns={'patient_volume': 'patient_volume_2021'})
    val.to_csv(val_path, index=False)

    return {
        'history_output_path': str(history_path),
        'train_output_path': str(train_path),
        'validation_output_path': str(val_path),
        'history_rows': int(len(history)),
        'train_rows': int((history['year'] < TARGET_YEAR).sum()),
        'validation_rows': int((history['year'] == TARGET_YEAR).sum()),
    }


In [ ]:
# =========================
# 6) QA report writers
# =========================

def _fmt_columns(cols: list[str]) -> str:
    # Make column lists readable in markdown.
    return '(none)' if not cols else ', '.join(cols)


def write_schema_summary(context: dict[str, Any]) -> Path:
    # This report lists each input file and the columns we detected.
    path = QA_DIR / 'schema_summary.md'
    om16 = context.get('onmarg_2016_meta', {})
    om21 = context.get('onmarg_2021_meta', {})
    cw = context.get('crosswalk_meta', {})
    pt = context.get('patients_meta', {})

    lines = [
        '# Schema Summary',
        '',
        '## ON-Marg 2016',
        f"- Source: `{om16.get('source_path', 'not found')}`",
        f"- Sheet: `{om16.get('sheet_name', 'n/a')}`",
        f"- Detected columns: {_fmt_columns(om16.get('detected_columns', []))}",
        '',
        '## ON-Marg 2021',
        f"- Source: `{om21.get('source_path', 'not found')}`",
        f"- Sheet: `{om21.get('sheet_name', 'n/a')}`",
        f"- Detected columns: {_fmt_columns(om21.get('detected_columns', []))}",
        '',
        '## Crosswalk',
        f"- Source: `{cw.get('source_path', 'not found')}`",
        f"- Detected columns: {_fmt_columns(cw.get('detected_columns', []))}",
        '',
        '## Patients',
        f"- Source: `{pt.get('source_path', 'not found')}`",
        f"- Detected columns: {_fmt_columns(pt.get('detected_columns', []))}",
    ]

    # Write the markdown file so inputs are auditable.
    path.write_text('\n'.join(lines), encoding='utf-8')
    return path


def write_qa_report(context: dict[str, Any]) -> Path:
    # This is the main QA report for row counts, mapping rates, and blockers.
    path = QA_DIR / 'qa_report.md'

    blockers = context.get('blockers', [])
    warnings = context.get('warnings', [])
    mm = context.get('mapping_meta', {})
    ml = context.get('ml_base_meta', {})
    pm = context.get('patients_meta', {})
    cm = context.get('crosswalk_meta', {})
    coef = context.get('deprivation_coeff_meta', {})
    temp = context.get('temporal_data_meta', {})

    lines = [
        '# QA Report',
        '',
        f"Run timestamp: {datetime.now().isoformat(timespec='seconds')}",
        f"Target year: {TARGET_YEAR}",
        '',
        '## Input Row Counts',
        f"- ON-Marg 2016 rows: {context.get('onmarg_2016_meta', {}).get('row_count', 0)}",
        f"- ON-Marg 2021 rows: {context.get('onmarg_2021_meta', {}).get('row_count', 0)}",
        f"- Crosswalk rows (clean): {cm.get('row_count', 0)}",
        f"- Patient rows (clean): {pm.get('row_count', 0)}",
        '',
        '## Mapping Quality',
        f"- Mapping rate (rows): {mm.get('row_mapping_rate', 0.0):.2%}",
        f"- Mapping rate (patient_count): {mm.get('patient_count_mapping_rate', 0.0):.2%}",
        f"- Unmapped postal codes: {mm.get('unmapped_postal_codes', 0)}",
        '',
        '## Final Output',
        f"- Final DAUID row count: {ml.get('row_count', 0)}",
        f"- Patient source type: `{pm.get('source_type', 'unknown')}`",
        f"- Patient source file: `{pm.get('source_path', 'not found')}`",
        '',
        '## Crosswalk Mode',
        f"- Is synthetic crosswalk: {cm.get('is_synthetic', False)}",
        f"- Is approximate crosswalk: {cm.get('is_approximate', False)}",
        f"- Approx method: `{cm.get('approx_method', 'n/a')}`",
        '',
        '## Deprivation Coefficient',
        f"- Coefficient type: `{coef.get('coefficient_type', 'not_generated')}`",
        f"- Gamma estimate: {coef.get('gamma_estimate', 'n/a')}",
        f"- Why not true (if demo): {coef.get('reason_not_true', 'n/a')}",
        '',
        '## Temporal Outputs',
        f"- Train (2016-2020): `{temp.get('train_output_path', 'n/a')}`",
        f"- Holdout (2021): `{temp.get('validation_output_path', 'n/a')}`",
        '',
        '## Warnings',
    ]

    # Add warning and blocker sections.
    lines.extend([f'- {w}' for w in warnings] if warnings else ['- None'])
    lines.extend(['', '## Blockers'])
    lines.extend([f'- {b}' for b in blockers] if blockers else ['- None'])

    # Track any assumptions (for example, missing patient_count).
    assumptions = pm.get('assumptions', [])
    lines.extend(['', '## Assumptions Applied'])
    lines.extend([f'- {a}' for a in assumptions] if assumptions else ['- None'])

    path.write_text('\n'.join(lines), encoding='utf-8')
    return path


def write_deprivation_coefficient_report(context: dict[str, Any]) -> Path:
    # Separate report focused on the gamma coefficient.
    path = QA_DIR / 'deprivation_coefficient_report.md'
    coef = context.get('deprivation_coeff_meta', {})
    temp = context.get('temporal_data_meta', {})

    lines = [
        '# Deprivation Coefficient Report',
        '',
        'This explains if the coefficient is true or demo and why.',
        '',
        f"- Coefficient type: `{coef.get('coefficient_type', 'not_generated')}`",
        f"- Gamma estimate: {coef.get('gamma_estimate', 'n/a')}",
        f"- Regression slope: {coef.get('slope', 'n/a')}",
        f"- Regression intercept: {coef.get('intercept', 'n/a')}",
        f"- Rows used: {coef.get('rows_used', 'n/a')}",
        f"- Reason not true (if demo): {coef.get('reason_not_true', 'n/a')}",
        '',
        f"- Train output: `{temp.get('train_output_path', 'n/a')}`",
        f"- Validation output: `{temp.get('validation_output_path', 'n/a')}`",
    ]

    path.write_text('\n'.join(lines), encoding='utf-8')
    return path


In [ ]:
# =========================
# 7) Orchestrator
# =========================

def run_pipeline() -> tuple[dict[str, Any], int]:
    # Central context used by all steps to share metadata, warnings, and blockers.
    context: dict[str, Any] = {'blockers': [], 'warnings': []}

    # 1) Always create template/placeholder patient files so the pipeline is runnable.
    context['patient_templates'] = ensure_patient_templates()

    # 2) ON-Marg: locate raw files and clean each census year.
    onmarg_files = discover_onmarg_files()
    context['onmarg_files'] = {k: (str(v) if v else None) for k, v in onmarg_files.items()}

    onmarg_2016 = None
    onmarg_2021 = None

    for year in [2016, 2021]:
        path = onmarg_files.get(year)
        if not path:
            context['blockers'].append(f'ON-Marg {year} file not found')
            continue
        try:
            # Load raw spreadsheet and then map columns to canonical names.
            raw, meta = load_onmarg_year(path, year)
            clean, clean_meta = clean_onmarg(raw, year)
            meta.update(clean_meta)
            context[f'onmarg_{year}_meta'] = meta

            # Store clean data for downstream steps.
            if year == 2016:
                onmarg_2016 = clean
            else:
                onmarg_2021 = clean
        except Exception as e:
            context['blockers'].append(str(e))

    # 3) Patients: pick best available file (real > synthetic > placeholder).
    patients_df = None
    source_type = 'placeholder'
    try:
        patient_path, source_type = discover_patient_file()
        patients_df, patients_meta = load_and_clean_patients(patient_path, source_type)
        context['patients_meta'] = patients_meta
    except Exception as e:
        context['blockers'].append(str(e))

    # 4) Crosswalk: real file > approximate boundary join > synthetic fallback.
    crosswalk_df = None
    cw_path = discover_crosswalk_file()
    try:
        if cw_path:
            # Use real crosswalk if present.
            crosswalk_df, cw_meta = load_and_clean_crosswalk(cw_path)

            # If the crosswalk is synthetic, rebuild it using observed patient postals.
            if cw_meta.get('is_synthetic') and ALLOW_SYNTHETIC_WORKAROUNDS and onmarg_2021 is not None and patients_df is not None:
                crosswalk_df, cw_meta = generate_synthetic_crosswalk(onmarg_2021, observed_postals=patients_df['postal_code'])
                context['warnings'].append('Rebuilt synthetic crosswalk from detected patient postal codes.')
        else:
            # Try approximate boundary + points method if files exist.
            boundary_zip = discover_da_boundary_zip(TARGET_YEAR) or discover_da_boundary_zip(2016)
            points_file = discover_points_file()

            # If no points file is found, try to download the public geoinfo dataset.
            if boundary_zip and points_file is None:
                points_file = download_geoinfo_points_file()
                if points_file:
                    context['warnings'].append('Downloaded geoinfo points dataset for approximate crosswalk.')

            if boundary_zip and points_file:
                crosswalk_df, cw_meta = build_crosswalk_from_boundaries_and_points(boundary_zip, points_file)
                context['warnings'].append('Crosswalk built from boundary + points (approximate, demo-only).')
            elif ALLOW_SYNTHETIC_WORKAROUNDS and onmarg_2021 is not None:
                observed_postals = patients_df['postal_code'] if patients_df is not None else None
                crosswalk_df, cw_meta = generate_synthetic_crosswalk(onmarg_2021, observed_postals=observed_postals)
                context['warnings'].append('Crosswalk missing; generated synthetic crosswalk.')
            else:
                raise RuntimeError('Crosswalk missing and synthetic fallback disabled.')

        context['crosswalk_meta'] = cw_meta
    except Exception as e:
        context['blockers'].append(str(e))

    # 5) Map patients to DAUID and aggregate 2021 patient volume.
    patient_volume_df = None
    if patients_df is not None and crosswalk_df is not None and onmarg_2021 is not None:
        mapped_df, map_meta = map_patients_to_dauid(patients_df, crosswalk_df)
        context['mapping_meta'] = map_meta

        patient_volume_df, pv_meta = aggregate_patient_volume(mapped_df)
        context['patient_volume_meta'] = pv_meta

        # Merge ON-Marg 2021 + patient volume to make ML base.
        _, ml_meta = build_ml_base_2021(onmarg_2021, patient_volume_df)
        context['ml_base_meta'] = ml_meta

    # 6) Coefficient + synthetic temporal data (only if we have all required inputs).
    if patient_volume_df is not None and onmarg_2016 is not None and onmarg_2021 is not None:
        crosswalk_meta = context.get('crosswalk_meta', {})
        crosswalk_is_real = bool(
            crosswalk_df is not None
            and not crosswalk_meta.get('is_synthetic', False)
            and not crosswalk_meta.get('is_approximate', False)
        )
        patient_is_real = bool(context.get('patients_meta', {}).get('used_real_patient_file', False))
        no_assumptions = len(context.get('patients_meta', {}).get('assumptions', [])) == 0

        # Mark coefficient as true only if all conditions are satisfied.
        is_true = crosswalk_is_real and patient_is_real and no_assumptions
        reasons = []
        if not crosswalk_is_real:
            reasons.append('Crosswalk is synthetic or approximate.')
        if not patient_is_real:
            reasons.append('Patient source is not designated final ORN label source.')
        if not no_assumptions:
            reasons.append('Patient data required default assumptions (missing count/year).')

        reason_not_true = 'All true-data conditions satisfied.' if not reasons else ' '.join(reasons)
        if reasons:
            context['warnings'].append('Deprivation coefficient is demo-only because true-data conditions are not fully met.')

        # Estimate gamma (deprivation coefficient).
        gamma, coef_meta = estimate_deprivation_coefficient(
            onmarg_2021,
            patient_volume_df,
            is_true=is_true,
            reason_not_true=reason_not_true,
        )
        context['deprivation_coeff_meta'] = coef_meta

        # Use gamma to backcast 2016-2020 training data.
        temporal_meta = generate_temporal_training_data(onmarg_2016, onmarg_2021, patient_volume_df, gamma)
        context['temporal_data_meta'] = temporal_meta

    # 7) Always write QA reports (even if blockers exist).
    write_schema_summary(context)
    write_deprivation_coefficient_report(context)
    write_qa_report(context)

    return context, (1 if context['blockers'] else 0)


In [ ]:
# =========================
# 8) Run pipeline
# =========================

context, exit_code = run_pipeline()
print('Pipeline run complete')
print('Exit code:', exit_code)
print('Blockers:', len(context.get('blockers', [])))
print('Warnings:', len(context.get('warnings', [])))

if context.get('warnings'):
    print('\nWarnings:')
    for w in context['warnings']:
        print('-', w)

if context.get('blockers'):
    print('\nBlockers:')
    for b in context['blockers']:
        print('-', b)

print('\nMain output:', context.get('ml_base_meta', {}).get('output_path', 'n/a'))
print('QA report:', str(QA_DIR / 'qa_report.md'))


## What to tell your team (copy-ready)

- The data pipeline is fully runnable in one notebook.
- ON-Marg 2016/2021 cleaning, patient linkage, DAUID aggregation, QA reporting, and synthetic history generation are complete.
- The deprivation-coefficient regression step is implemented.
- The coefficient is marked `demo` if inputs are not fully validation-grade (for example synthetic crosswalk or missing patient count/year fields).
- Therefore, current outputs are suitable for coursework demo/prototyping, not final real-world validation claims.
